# YannET · Diagnóstico inicial do SAM 3

Este notebook usa gravações da C1 e da CAM 16 para diagnosticar a segmentação por instância. O ensaio de amontoamentos na foto da CAM 7 está em `explorar_grupos.py`.

Compare `chicken` e `chick` em quadro inteiro e em quatro recortes sobrepostos. O teste rápido processa um quadro de desenvolvimento. A comparação completa processa 12.

**Local:** use o kernel do ambiente virtual do YannET e prepare os quadros com `PYTHONPATH=src python -m yannet.quadros`.

**Colab:** no menu **Ambiente de execução**, escolha um tipo de ambiente com GPU. Execute a célula abaixo e selecione o arquivo `sam3-piloto-colab.zip` gerado pelo YannET. Ele contém o código e os quadros de desenvolvimento. A gravação reservada da CAM 7 fica no computador.

In [ ]:
import io
import subprocess
import sys
import tempfile
from pathlib import Path
from zipfile import ZipFile

try:
    from google.colab import files
except ImportError:
    files = None

EM_COLAB = files is not None
if EM_COLAB:
    PASTA = Path(tempfile.mkdtemp(prefix="yannet-"))
    enviados = files.upload()
    if len(enviados) != 1 or not next(iter(enviados)).endswith(".zip"):
        raise ValueError("Selecione apenas o arquivo sam3-piloto-colab.zip.")
    with ZipFile(io.BytesIO(next(iter(enviados.values())))) as pacote:
        pacote.extractall(PASTA)
    MANIFESTO = PASTA / "quadros" / "quadros.json"
    RESULTADOS = Path.cwd() / "yannet-resultados"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PASTA / "requirements-sam3.txt")],
        check=True,
    )
else:
    atual = Path.cwd()
    candidatas = [atual, atual / "src" / "yannet"]
    PASTA = next((p for p in candidatas if (p / "comparar_sam3.py").is_file()), None)
    if PASTA is None:
        raise FileNotFoundError("Abra o notebook a partir da raiz do repositório ou de src/yannet.")
    MANIFESTO = PASTA / "outputs" / "desenvolvimento" / "quadros.json"
    RESULTADOS = PASTA / "outputs"

if not MANIFESTO.is_file():
    raise FileNotFoundError(f"Manifesto ausente em {MANIFESTO}. Prepare os quadros primeiro.")
print(MANIFESTO)

## Confirmar o acesso ao modelo

Solicite acesso na [página do SAM 3](https://huggingface.co/facebook/sam3). Após a liberação, use um [token de leitura do Hugging Face](https://huggingface.co/settings/tokens). No computador, autentique no terminal com `hf auth login`. No Colab, a célula pede o token de forma oculta se ainda não houver sessão.

In [ ]:
import os
from getpass import getpass

from huggingface_hub import HfApi, get_token

token = get_token()
if token is None and EM_COLAB:
    token = getpass("Token de leitura do Hugging Face: ").strip()
if not token:
    raise RuntimeError("Autentique com hf auth login e execute esta célula novamente.")
HfApi().auth_check("facebook/sam3", repo_type="model", token=token)
if EM_COLAB:
    os.environ["HF_TOKEN"] = token
del token
print("Acesso de leitura confirmado.")

In [ ]:
from datetime import datetime, timezone

COMPARACAO_COMPLETA = False  # Mude para True depois de revisar o teste rápido.
DISPOSITIVO = "cuda" if EM_COLAB else "cpu"
execucao = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SAIDA = RESULTADOS / f"sam3-{DISPOSITIVO}-{execucao}"
comando = [
    sys.executable,
    "-u",
    str(PASTA / "comparar_sam3.py"),
    "--manifesto",
    str(MANIFESTO),
    "--saida",
    str(SAIDA),
    "--dispositivo",
    DISPOSITIVO,
]
if not COMPARACAO_COMPLETA:
    comando += ["--limite", "1", "--modo", "quadro", "--prompt", "chicken"]
ambiente = dict(os.environ)
if not EM_COLAB:
    ambiente.setdefault("HF_HUB_CACHE", str(RESULTADOS / "cache-hf"))
subprocess.run(comando, env=ambiente, check=True)

In [ ]:
import csv
import json

from IPython.display import Image, display

print(json.dumps(json.loads((SAIDA / "execucao.json").read_text()), indent=2, ensure_ascii=False))
with (SAIDA / "comparacao.csv").open(newline="", encoding="utf-8") as arquivo:
    linhas = list(csv.DictReader(arquivo))
primeiro = linhas[0]["quadro"]
display(Image(filename=str(MANIFESTO.parent / primeiro)))
for linha in linhas:
    if linha["quadro"] == primeiro:
        print(linha["prompt"], linha["modo"], f"{linha['tempo_s']} s")
        display(Image(filename=str(SAIDA / linha["previa"])))

In [ ]:
# No Colab, execute esta célula para guardar os resultados no computador.
if EM_COLAB:
    import shutil

    arquivo = shutil.make_archive(str(SAIDA), "zip", root_dir=SAIDA)
    files.download(arquivo)